In [1]:
from osgeo import gdal
from osgeo import gdal_array
from osgeo import osr

In [2]:
# SYS Libraries ##########################################################################################################
import sys
import os
import subprocess
import glob

# Basic Libraries ##########################################################################################################
import numpy as np
import pandas as pd
#import matplotlib.pyplot as plt
#import matplotlib.image as mpimg

from IPython.display import display

# Image Manipulation Libraries ##########################################################################################################
#import shapely 
import csv

#import tifffile as tiff

#from PIL import Image, ImageFilter, ImageEnhance, ImageOps, ImageChops

# GDAL Libraries ##########################################################################################################
from osgeo import gdal
from osgeo import gdal_array
from osgeo import osr

# Keras Libraries ##########################################################################################################
from keras.models import Model, load_model

from keras import backend as K

from sklearn.metrics import jaccard_similarity_score

from collections import defaultdict

%matplotlib inline

# Basic DATA ##########################################################################################################
#Image data
root_folder = os.path.abspath("")
train_dir = os.path.join(root_folder, '0_traindata/')
baseimage = 'tif'
codimage = '{}.'+ baseimage

#Train Data info
# Basic DAta: Train points/polygons data
trainxy_folder = os.path.abspath("1_trainshapes/")

modelo = 'modelo'
gridimages = modelo + '_' + 'grid_extent_full.csv'
predimages = modelo + '_' + 'predict_extent_full.csv'
fishnetfile = modelo + '_' + 'fishnet_grid_extent.csv'
trainxy = modelo + '_' + 'xy_deadtree.csv'

#Template Matching Output
tempmatch_out_folder = os.path.abspath("y_outCNN/")


print(tempmatch_out_folder)

from keras import backend as K
K.tensorflow_backend._get_available_gpus()

Using TensorFlow backend.


C:\Users\snojek\Documents\0-Model_Development\1_MdP_Plantacion\y_outCNN


[]

In [3]:
###################################################################################
## Predict output for target Image
def jaccard_coef(y_true, y_pred):
    smooth = 1e-12
    # __author__ = Vladimir Iglovikov
    intersection = K.sum(y_true * y_pred, axis=[0, -1, -2])
    sum_ = K.sum(y_true + y_pred, axis=[0, -1, -2])

    jac = (intersection + smooth) / (sum_ - intersection + smooth)

    return K.mean(jac)


def jaccard_coef_int(y_true, y_pred):
    smooth = 1e-12
    # __author__ = Vladimir Iglovikov
    y_pred_pos = K.round(K.clip(y_pred, 0, 1))

    intersection = K.sum(y_true * y_pred_pos, axis=[0, -1, -2])
    sum_ = K.sum(y_true + y_pred, axis=[0, -1, -2])
    jac = (intersection + smooth) / (sum_ - intersection + smooth)
    return K.mean(jac)


def loadmodel():
    model = load_model('4_training_model/model.h5', custom_objects={'jaccard_coef': jaccard_coef, 'jaccard_coef_int': jaccard_coef_int})
    return model

def loadarray(inputname):
    numpy_folder = os.path.abspath("3_train_array/")
    train_input_file = numpy_folder + '/' + inputname + ".npy"
    train_x_data = np.load(train_input_file)
    return train_x_data

def read_and_normalize_train_data(train_data, norm_mean, norm_std):
    #open array data
    train_data = ((train_data[:,:,:] - norm_mean)/norm_std).astype(np.float32)
    return train_data


def loadimage(IMagenID):
    print("Process 1 - Load image: ", IMagenID)
    #Load bands
    file = train_dir + codimage .format(IMagenID)
    cube = gdal.Open(file)
    
    bnd1 = cube.GetRasterBand(1)
    img1 = bnd1.ReadAsArray(0,0,cube.RasterXSize, cube.RasterYSize)
    del bnd1
    
    bnd2 = cube.GetRasterBand(2)
    img2 = bnd2.ReadAsArray(0,0,cube.RasterXSize, cube.RasterYSize)
    del bnd2
    
    bnd3 = cube.GetRasterBand(3)
    img3 = bnd3.ReadAsArray(0,0,cube.RasterXSize, cube.RasterYSize) 
    del bnd3

    #Get image corners...with GDAL
    ulx, xres, xskew, uly, yskew, yres  = cube.GetGeoTransform()
    lrx = ulx + (cube.RasterXSize * xres)
    lry = uly + (cube.RasterYSize * yres)
    del cube
    
    #Stack an image to 3 bands/display
    a = np.array(img1)
    del img1
    b = np.array(img2)
    del img2
    c = np.array(img3)
    del img3

    img = np.dstack((a,b,c))
    del a
    del b
    del c
    
    #im_rgb = np.array(img, np.uint8)
    im_size = img.shape[:2]
    

    xfull_max = float(lrx)
    xfull_min = float(ulx)
    yfull_max = float(uly)
    yfull_min = float(lry)

    print("XMax", xfull_max, "XMin", xfull_min, "YMax", yfull_max, "YMin", yfull_min)
    print("Input image size: ", im_size)
    print("Process 1 - DONE")
    
    return img, im_size, xfull_max, xfull_min, yfull_max, yfull_min
    

def get_scalers(im_size, xfull_max, xfull_min, yfull_max, yfull_min):
    h, w = im_size
    w_ = 1 * w * w / (w + 1)
    h_ = 1 * h * h / (h + 1)
    wi_max = abs(xfull_max - xfull_min)
    he_max = abs(yfull_max - yfull_min)
    return w_ / wi_max, h_ / he_max, w, h

def pred_image(vary, varx, h, w, img, bands_out, input_bands, output_bands):
    print("Process 2 - Generate image data: ", IMagenID)
    lastrow = int(h - vary)
    lastcol = int(w - varx)
    print("Last Row ", lastrow/vary)
    print("Last Col ", lastcol/varx)

    targetimage=[]
    minx = 0

    a = 0
    b = lastcol+1
    c = 0
    d = lastrow+1

    #Generate prediction image
    prediction = np.zeros((h+vary, w+vary, output_bands), dtype=np.uint8)
    
    minx = a
    norm_mean = loadarray('normdata_meanarray')
    norm_std = loadarray('normdata_stdarray')

    for i in range(a,b,varx):
            maxx = int(i + varx)
            miny = c
            for j in range(c, d, vary):
                maxy = int(miny + vary)
                if(minx <= lastcol and miny <= lastrow):
                    imagetarget = img[miny:maxy, minx:maxx]
                    imagetarget = np.reshape(imagetarget, [vary, varx, 3])
                    imagetarget = read_and_normalize_train_data(imagetarget,norm_mean,norm_std)
                    imagetarget = np.reshape(imagetarget, [1, vary, varx, input_bands])
                    
                    prediction[miny:maxy, minx:maxx] = 255 * model.predict(imagetarget, batch_size=1, verbose=0)
                    
                      
                miny += vary
            minx += varx
    
    del img
    
    print("Process 2 - DONE")    
    return prediction


######################################################################################################
#Process image for better results. Enhace image and SAVE Results


def savetiff(src, prediction, xfull_min,yfull_min,xfull_max,yfull_max, h, w, IMagenID, clases):
    #Define image size & data
    xmin,ymin,xmax,ymax = [xfull_min,yfull_min,xfull_max,yfull_max]
    nrows = h
    ncols = w
    xres = (xmax-xmin)/float(ncols)
    yres = (ymax-ymin)/float(nrows)
    geotransform=(xmin,xres,0,ymax,0, -yres)   

    name = 'y_outCNN/fase1_' + str(IMagenID) + '_v0.tiff'
    
    output_raster = gdal.GetDriverByName('GTiff').Create(name ,ncols, nrows, clases, gdal.GDT_Byte,options=['COMPRESS=LZW'])  # Open the file
    output_raster.SetGeoTransform(geotransform)  
    srs = osr.SpatialReference()                 
    srs.ImportFromEPSG(src)                     
    output_raster.SetProjection( srs.ExportToWkt() )   

    output_raster.GetRasterBand(1).WriteArray(prediction[:h,:w,0])

    output_raster = None



In [4]:
#Lets try to read the grid file and get all the images names

#Load model and weights
model = loadmodel()

#Load the target images 
GS = pd.read_csv(trainxy_folder + '/' + predimages, names=['id', 'ImageId'], skiprows=1)
FS = pd.DataFrame(GS)
count = 1
first = 0

#Loop target images
for imagetarget in FS.iterrows():
    idimage = imagetarget[1][0]
    IMagenID = imagetarget[1][1].strip()
    print("Load data from: ", IMagenID, "Id: ", idimage)
    print("Image: ", count, " from ", len(FS))
    
    img, im_size, xfull_max, xfull_min, yfull_max, yfull_min = loadimage(IMagenID)
    x_scaler, y_scaler, w, h = get_scalers(im_size, xfull_max, xfull_min, yfull_max, yfull_min)
    
    vary = 250
    varx = 250
    clases = 1
    input_bands = 3
    output_bands = 1
    src = 32721
    
    print ("Start prediction")
    prediction = pred_image(vary, varx, h, w, img, clases, input_bands, output_bands)    
    del img
    
    print ("Saving image")
    savetiff(src, prediction, xfull_min,yfull_min,xfull_max,yfull_max, h, w, IMagenID, clases) 
   
    count = count + 1            
    
    del prediction
    
    print("-------------------------------------------------------------------------------------")

print(" ## Finish prediction ##")

Load data from:  El_Lucero_II_dam_JIB_utm_10cm Id:  121
Image:  1  from  1
Process 1 - Load image:  El_Lucero_II_dam_JIB_utm_10cm
XMax 470954.93168000004 XMin 469847.33168000006 YMax 6306811.99179 YMin 6305106.99179
Input image size:  (17050, 11076)
Process 1 - DONE
Start prediction
Process 2 - Generate image data:  El_Lucero_II_dam_JIB_utm_10cm
Last Row  67.2
Last Col  43.304
Process 2 - DONE
Saving image
-------------------------------------------------------------------------------------
 ## Finish prediction ##
